<a href="https://colab.research.google.com/github/toche7/AnalyticCases/blob/main/prepDatafromLog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
# เชื่อมต่อกับ Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


### วิธีการเรียกใช้ไฟล์จาก Google Drive

หลังจากเชื่อมต่อสำเร็จแล้ว คุณสามารถเปลี่ยนเส้นทางไฟล์ (Path) ในสคริปต์ของคุณให้ชี้ไปยัง Google Drive ได้โดยตรง เช่น:

*   **ไฟล์ Log:** `Path('/content/drive/MyDrive/โฟลเดอร์ของคุณ/ezp_aug01-03_2026.log')`
*   **ไฟล์ Mapping:** `Path('/content/drive/MyDrive/โฟลเดอร์ของคุณ/new_mapping_file.md')`

In [2]:
#!/usr/bin/env python3
"""Stream EZproxy logs into a 35-column, ezPAARSE-shaped candidate CSV.

The Markdown mapping identifies platforms and their permitted resource types.
URL rules identify *candidate* consultations; this is not ezPAARSE or COUNTER.
Unidentified or ambiguous URLs are excluded and reported, never guessed.
"""

import argparse
import csv
import gzip
import hashlib
import ipaddress
import re
import sys
from collections import Counter
from datetime import datetime
from pathlib import Path
from urllib.parse import unquote, urlsplit

FIELDS = (
    'datetime date login platform_name publisher_name rtype mime '
    'print_identifier online_identifier title_id doi publication_title '
    'publication_date unitid domain on_campus log_id ezpaarse_version '
    'ezpaarse_date middlewares_version middlewares_date platforms_version '
    'platforms_date middlewares id pii sudoc-ppn title type subject '
    'geoip-country geoip-latitude geoip-longitude host url'
).split()

# Verified against the supplied EZproxy lines: %h %l %u %t "%r" %s %b.
LOG = re.compile(
    r'^\s*(?P<host>\S+)\s+\S+\s+(?P<login>\S+)\s+'
    r'\[(?P<timestamp>[^]]+)\]\s+'
    r'"(?P<method>[A-Z]+)\s+(?P<url>\S+)\s+[^\s"]+"\s+'
    r'(?P<status>\d{3})\s+(?:\d+|-)(?:\s+.*)?$'
)
DOI = re.compile(r'10\.\d{4,9}/[^\s?#&]+', re.I)
STATIC = re.compile(r'\.(?:css|js|mjs|png|jpe?g|gif|svg|ico|woff2?|ttf|map)(?:$|[?#])', re.I)
SD_PII = re.compile(r'^/science/article/pii/([a-z0-9]+)', re.I)


def load_mapping(path):
    """Load the supplied 5-column Markdown table; keep every allowed rType."""
    platforms = {}
    for number, line in enumerate(path.read_text(encoding='utf-8-sig').splitlines(), 1):
        if not line.startswith('|'):
            continue
        cells = [cell.strip() for cell in line.strip().strip('|').split('|')]
        if len(cells) != 5 or cells[0] == 'Platform-show' or all(set(c) <= {'-', ':'} for c in cells):
            continue
        shown, name, rtype, domain, _note = cells
        domain = domain.lower().rstrip('.')
        if not (shown and name and rtype and domain):
            raise ValueError(f'Incomplete mapping row {number}')
        allowed = set(rtype.split('/'))
        if any(not re.fullmatch(r'[A-Z_]+', kind) for kind in allowed):
            raise ValueError(f'Invalid rType on mapping row {number}: {rtype}')
        if domain in platforms and platforms[domain]['name'] != name:
            raise ValueError(f'Conflicting platform names for {domain}')
        item = platforms.setdefault(domain, {'name': name, 'allowed': set(), 'shown': {}})
        item['allowed'].update(allowed)
        for kind in allowed:
            item['shown'].setdefault(kind, set()).add(shown)
    if not platforms:
        raise ValueError('No platform rows found in Markdown mapping')
    return platforms


def open_log(path):
    opener = gzip.open if path.suffix.lower() == '.gz' else open
    return opener(path, 'rt', encoding='utf-8', errors='replace')


def classify(domain, path):
    """Conservative URL hints; the mapping checks whether rType is permitted."""
    lower = path.lower()
    if domain == 'www.sciencedirect.com':
        if lower == '/search' or lower.startswith('/search/'):
            return 'SEARCH', '', ''
        match = SD_PII.match(path)
        if match:
            return 'ARTICLE', match.group(1), match.group(1)
    if domain in ('www.scopus.com', 'scifinder.cas.org', 'jcr.clarivate.com'):
        if '/search' in lower:
            return 'SEARCH', '', ''
        # JCR ANALYSIS requires verified URL examples.
        return None
    if domain == 'learning.oreilly.com':
        if '/videos/' in lower or '/video/' in lower:
            return 'VIDEO', '', ''
        if '/library/view/' in lower:
            return 'BOOK_SECTION', '', ''
    if '/chapter/' in lower or '/book-section/' in lower or '/book/chapter/' in lower:
        return 'BOOK_SECTION', '', ''
    if '/science/article/' in lower or '/article/' in lower or '/articles/' in lower or '/fulltext' in lower:
        return 'ARTICLE', '', ''
    # Broad /doi/ and /book/ patterns can refer to several rTypes.
    return None


def extract(line, platforms, networks=()):
    raw = line.rstrip('\r\n')
    match = LOG.match(raw)
    if not match:
        return None, 'unparsed'
    data = match.groupdict()
    if data['method'] != 'GET' or not data['status'].startswith('2'):
        return None, 'http_filtered'
    parts = urlsplit(data['url'])
    if parts.scheme.lower() not in ('http', 'https') or not parts.hostname:
        return None, 'unparsed'
    domain = parts.hostname.lower().rstrip('.')
    if STATIC.search(parts.path) or domain in ('ejournal.mahidol.ac.th', 'www.li.mahidol.ac.th'):
        return None, 'nonresource'
    platform = platforms.get(domain)
    if platform is None:
        return None, 'unmapped_domain'
    decoded_path = unquote(parts.path)
    resource = classify(domain, decoded_path)
    if resource is None:
        return None, 'unrecognized_url'
    rtype, unitid, pii = resource
    if rtype not in platform['allowed']:
        return None, 'unmapped_rtype'
    try:
        timestamp = datetime.strptime(data['timestamp'], '%d/%b/%Y:%H:%M:%S %z')
    except ValueError:
        return None, 'unparsed'
    pdf = decoded_path.lower().endswith(('.pdf', '/pdf', '/pdfft')) or '/pdf/' in decoded_path.lower()
    doi = DOI.search(decoded_path)
    event = dict.fromkeys(FIELDS, '')
    event.update({
        'datetime': timestamp.isoformat(), 'date': timestamp.date().isoformat(),
        'login': '' if data['login'] == '-' else data['login'],
        'platform_name': platform['name'], 'rtype': rtype,
        'mime': 'PDF' if pdf else 'HTML',
        'doi': doi.group(0).rstrip('.,;') if doi else '',
        'unitid': unitid, 'pii': pii, 'domain': domain,
        'log_id': hashlib.sha1(raw.encode('utf-8')).hexdigest(),
        'host': data['host'], 'url': data['url'],
    })
    if networks:
        try:
            ip = ipaddress.ip_address(data['host'])
            event['on_campus'] = 'Y' if any(ip in net for net in networks) else 'N'
        except ValueError:
            pass
    return event, 'event'


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('log', type=Path, help='EZproxy log, plain text or .gz')
    parser.add_argument('output', type=Path, help='35-column candidate CSV')
    parser.add_argument('--mapping', required=True, type=Path,
                        help='Platform / Resource Type / Domain Mapping Markdown file')
    parser.add_argument('--campus-cidr', action='append', default=[], metavar='CIDR',
                        help='Use only if the first log IP is verified as the client IP')

    # --- START: User defined arguments --- #
    # ใช้ Log จาก Google Drive แต่ใช้ Mapping จาก Local Colab ซึ่งดาวน์โหลดมาเรียบร้อยแล้ว
    log_file_path = Path('/content/drive/MyDrive/EZproxyData/ezp_aug01-03_2026.log')
    output_file_path = Path('/content/extracted_events_v2.csv')
    mapping_file_path = Path('/content/drive/MyDrive/EZproxyData/platform_resource_domain_mapping.md')
    # --- END: User defined arguments --- #

    # Pass the user-defined file paths to parse_args
    args = parser.parse_args([str(log_file_path), str(output_file_path), '--mapping', str(mapping_file_path)])

    try:
        platforms = load_mapping(args.mapping)
        networks = [ipaddress.ip_network(value) for value in args.campus_cidr]
    except (OSError, ValueError) as error:
        parser.error(str(error))
    counts = Counter()
    with open_log(args.log) as source, args.output.open('w', newline='', encoding='utf-8') as target:
        writer = csv.DictWriter(target, fieldnames=FIELDS)
        writer.writeheader()
        for line in source:
            event, reason = extract(line, platforms, networks)
            counts[reason] += 1
            if event is not None:
                writer.writerow(event)
    print(' '.join(f'{key}={counts[key]}' for key in (
        'event', 'unparsed', 'http_filtered', 'nonresource',
        'unmapped_domain', 'unmapped_rtype', 'unrecognized_url')),
        file=sys.stderr)
    print(f'mapping_domains={len(platforms)}', file=sys.stderr)


if __name__ == '__main__':
    main()

event=4703 unparsed=46 http_filtered=151458 nonresource=181885 unmapped_domain=63497 unmapped_rtype=2026 unrecognized_url=43284
mapping_domains=16


In [3]:
import pandas as pd
import os
from io import StringIO

# 1. โหลดข้อมูลสกัดใหม่จาก '/content/extracted_events_v2.csv'
direct_df = pd.read_csv('/content/extracted_events_v2.csv')

# 2. จัดระเบียบข้อมูลและลบช่องว่าง
direct_df['domain'] = direct_df['domain'].astype(str).str.strip()
direct_df['rtype'] = direct_df['rtype'].astype(str).str.strip()
direct_df['domain'] = direct_df['domain'].replace('scifinder-n.cas.org', 'scifinder.cas.org')

# 3. กู้คืนตัวแปร domain_rtype_to_platform_show_map ในกรณีที่หลุดจากหน่วยความจำ
if 'domain_rtype_to_platform_show_map' not in locals() and 'domain_rtype_to_platform_show_map' not in globals():
    print("⚠️ ไม่พบตัวแปร Mapping ในหน่วยความจำ ระบบกำลังโหลดและสร้างพจนานุกรม Mapping ใหม่จากไฟล์ให้โดยอัตโนมัติ...")
    mapping_file_path = '/content/drive/MyDrive/EZproxyData/platform_resource_domain_mapping.md'
    if os.path.exists(mapping_file_path):
        with open(mapping_file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        lines = content.split('\n')
        header_line, separator_line = None, None
        for i, line in enumerate(lines):
            if '| Platform-show | Platform-name | rType | Domain |' in line:
                header_line = i
            elif '|---|---|---|---|---' in line:
                separator_line = i
        if header_line is not None and separator_line is not None:
            table_lines = lines[header_line:separator_line] + lines[separator_line+1:]
            table_string = '\n'.join(table_lines).replace(' | ', ',').replace('|', '')
            m_df = pd.read_csv(StringIO(table_string))
            m_df.columns = m_df.columns.str.strip()
            m_df['Domain'] = m_df['Domain'].astype(str).str.strip()
            m_df['rType'] = m_df['rType'].astype(str).str.strip()
            m_df['Platform-show'] = m_df['Platform-show'].astype(str).str.strip()
            m_df_cleaned = m_df.drop_duplicates(subset=['Domain', 'rType'], keep='first')
            domain_rtype_to_platform_show_map = m_df_cleaned.set_index(['Domain', 'rType'])['Platform-show'].to_dict()
            print("✅ สร้างระบบ Mapping สำเร็จ!")
        else:
            raise ValueError("ไม่สามารถแปลงหัวข้อตารางในไฟล์ Mapping ได้")
    else:
        raise FileNotFoundError(f"ไม่พบไฟล์ Mapping ที่ {mapping_file_path} กรุณาดาวน์โหลดไฟล์นี้ก่อนใช้งาน")

# 4. จับคู่ (Mapping) ร่วมกับพจนานุกรม
direct_df['Platform-show'] = list(zip(direct_df['domain'], direct_df['rtype']))
direct_df['Platform-show'] = direct_df['Platform-show'].map(domain_rtype_to_platform_show_map)

# 5. ลบแถวที่ไม่สามารถระบุกลุ่มแพลตฟอร์มได้ออก
direct_df = direct_df.dropna(subset=['Platform-show']).copy()

# 6. ลบคอลัมน์ 'platform_name' ออกตามข้อกำหนดของคุณ
if 'platform_name' in direct_df.columns:
    direct_df = direct_df.drop(columns=['platform_name'])

# 7. บันทึกไฟล์ที่คลีนแล้วลงในระบบปลายทางบน Colab
cleaned_direct_output = '/content/08-2026_clean.csv'
direct_df.to_csv(cleaned_direct_output, index=False)

print(f"\n✅ ทำความสะอาดข้อมูลสำเร็จ!")
print(f"- จำนวนแถวทั้งหมดหลังคลีน: {direct_df.shape[0]} แถว")
print(f"- บันทึกไฟล์ผลลัพธ์พร้อมใช้งานไว้ที่: {cleaned_direct_output}")

⚠️ ไม่พบตัวแปร Mapping ในหน่วยความจำ ระบบกำลังโหลดและสร้างพจนานุกรม Mapping ใหม่จากไฟล์ให้โดยอัตโนมัติ...
✅ สร้างระบบ Mapping สำเร็จ!

✅ ทำความสะอาดข้อมูลสำเร็จ!
- จำนวนแถวทั้งหมดหลังคลีน: 4703 แถว
- บันทึกไฟล์ผลลัพธ์พร้อมใช้งานไว้ที่: /content/08-2026_clean.csv


In [4]:
direct_df

,datetime,date,login,publisher_name,rtype,mime,print_identifier,online_identifier,title_id,doi,...,sudoc-ppn,title,type,subject,geoip-country,geoip-latitude,geoip-longitude,host,url,Platform-show
0,2026-08-01T00:01:00+07:00,2026-08-01,Thaneeya.nan,NaN,ARTICLE,PDF,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.49.235.38,https://www.sciencedirect.com:443/science/arti...,ScienceDirect eJournals
1,2026-08-01T00:01:01+07:00,2026-08-01,Thaneeya.nan,NaN,ARTICLE,PDF,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.49.235.38,https://www.sciencedirect.com:443/science/arti...,ScienceDirect eJournals
2,2026-08-01T00:08:45+07:00,2026-08-01,u6506059,NaN,ARTICLE,PDF,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,223.207.219.216,https://www.nature.com:443/articles/s41419-024...,Nature eJournals
3,2026-08-01T00:08:46+07:00,2026-08-01,detchara.sum,NaN,SEARCH,HTML,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.145.158.87,https://www.scopus.com:443/pages/search/author...,Scopus
4,2026-08-01T00:08:47+07:00,2026-08-01,detchara.sum,NaN,SEARCH,HTML,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.145.158.87,https://www.scopus.com:443/gateway/search-mana...,Scopus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4698,2026-08-03T23:58:38+07:00,2026-08-03,sattawut.won,NaN,ARTICLE,HTML,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101.51.71.194,https://www.sciencedirect.com:443/science/arti...,ScienceDirect eJournals
4699,2026-08-03T23:58:39+07:00,2026-08-03,sattawut.won,NaN,ARTICLE,HTML,NaN,NaN,NaN,10.1016/j.jstrokecerebrovasdis.2012.10.003/201...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101.51.71.194,https://www.sciencedirect.com:443/article/rela...,ScienceDirect eJournals
4700,2026-08-03T23:58:39+07:00,2026-08-03,sattawut.won,NaN,ARTICLE,HTML,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,159.192.143.54,https://www.sciencedirect.com:443/sdfe/arp/iss...,ScienceDirect eJournals
4701,2026-08-03T23:58:41+07:00,2026-08-03,sattawut.won,NaN,ARTICLE,PDF,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101.51.71.194,https://www.sciencedirect.com:443/science/arti...,ScienceDirect eJournals


In [5]:
import pandas as pd

df = pd.read_csv('/content/extracted_events_v2.csv')
display(df.head())

,datetime,date,login,platform_name,publisher_name,rtype,mime,print_identifier,online_identifier,title_id,...,pii,sudoc-ppn,title,type,subject,geoip-country,geoip-latitude,geoip-longitude,host,url
0,2026-08-01T00:01:00+07:00,2026-08-01,Thaneeya.nan,Science Direct,NaN,ARTICLE,PDF,NaN,NaN,NaN,...,S0378517325001279,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.49.235.38,https://www.sciencedirect.com:443/science/arti...
1,2026-08-01T00:01:01+07:00,2026-08-01,Thaneeya.nan,Science Direct,NaN,ARTICLE,PDF,NaN,NaN,NaN,...,S0378517325001279,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.49.235.38,https://www.sciencedirect.com:443/science/arti...
2,2026-08-01T00:08:45+07:00,2026-08-01,u6506059,Nature Publishing Group,NaN,ARTICLE,PDF,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,223.207.219.216,https://www.nature.com:443/articles/s41419-024...
3,2026-08-01T00:08:46+07:00,2026-08-01,detchara.sum,Scopus,NaN,SEARCH,HTML,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.145.158.87,https://www.scopus.com:443/pages/search/author...
4,2026-08-01T00:08:47+07:00,2026-08-01,detchara.sum,Scopus,NaN,SEARCH,HTML,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.145.158.87,https://www.scopus.com:443/gateway/search-mana...


In [7]:
print(f"The DataFrame has {df.shape[0]} rows and {df.shape[1]} columns.")
print("Memory usage:")
df.info()

The DataFrame has 4703 rows and 35 columns.
Memory usage:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4703 entries, 0 to 4702
Data columns (total 35 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   datetime             4703 non-null   object 
 1   date                 4703 non-null   object 
 2   login                4703 non-null   object 
 3   platform_name        4703 non-null   object 
 4   publisher_name       0 non-null      float64
 5   rtype                4703 non-null   object 
 6   mime                 4703 non-null   object 
 7   print_identifier     0 non-null      float64
 8   online_identifier    0 non-null      float64
 9   title_id             0 non-null      float64
 10  doi                  830 non-null    object 
 11  publication_title    0 non-null      float64
 12  publication_date     0 non-null      float64
 13  unitid               1169 non-null   object 
 14  domain               4703 non-

### `df` DataFrame Column Definitions

Here's a breakdown of the columns in your `df` DataFrame:

*   **`datetime`**: The exact date and time of the log entry, including timezone information.
*   **`date`**: The date part of the log entry.
*   **`login`**: The user's login ID. This might be a username or a placeholder like '-' if not available.
*   **`publisher_name`**: The name of the publisher associated with the resource. (Appears to be largely null in this dataset).
*   **`rtype`**: The resource type (e.g., ARTICLE, SEARCH, BOOK_SECTION) as classified from the URL.
*   **`mime`**: The MIME type of the content, indicating if it's typically PDF or HTML.
*   **`print_identifier`**: An identifier for the print version of the resource. (Appears to be largely null).
*   **`online_identifier`**: An identifier for the online version of the resource. (Appears to be largely null).
*   **`title_id`**: An identifier for the title. (Appears to be largely null).
*   **`doi`**: Digital Object Identifier, a unique string used to identify content objects.
*   **`publication_title`**: The title of the publication. (Appears to be largely null).
*   **`publication_date`**: The date of publication. (Appears to be largely null).
*   **`unitid`**: A unique identifier for the specific unit or part of a resource.
*   **`domain`**: The domain name of the resource's URL.
*   **`on_campus`**: Indicates whether the access was from an on-campus IP address (Y/N). (Appears to be largely null in this dataset).
*   **`log_id`**: A unique hash ID generated for each raw log entry.
*   **`ezpaarse_version`**: Version of ezPAARSE. (Appears to be largely null).
*   **`ezpaarse_date`**: Date of ezPAARSE processing. (Appears to be largely null).
*   **`middlewares_version`**: Version of middlewares. (Appears to be largely null).
*   **`middlewares_date`**: Date of middlewares processing. (Appears to be largely null).
*   **`platforms_version`**: Version of platforms. (Appears to be largely null).
*   **`platforms_date`**: Date of platforms processing. (Appears to be largely null).
*   **`middlewares`**: Additional middleware information. (Appears to be largely null).
*   **`id`**: Generic identifier. (Appears to be largely null).
*   **`pii`**: Publisher Item Identifier, a proprietary identifier often used by publishers.
*   **`sudoc-ppn`**: Sudoc PPN (Numéro Permanent National), a unique identifier for bibliographic records in French academic libraries. (Appears to be largely null).
*   **`title`**: The title of the accessed content. (Appears to be largely null).
*   **`type`**: The type of resource. (Appears to be largely null).
*   **`subject`**: The subject of the resource. (Appears to be largely null).
*   **`geoip-country`**: Geographical country based on IP. (Appears to be largely null).
*   **`geoip-latitude`**: Geographical latitude based on IP. (Appears to be largely null).
*   **`geoip-longitude`**: Geographical longitude based on IP. (Appears to be largely null).
*   **`host`**: The IP address of the client accessing the resource.
*   **`url`**: The full URL of the accessed resource.
*   **`Platform-show`**: The standardized platform name derived from the `domain` and `rtype` using the provided mapping file.

In [6]:
from io import StringIO
import pandas as pd

# The markdown content is already in the 'content' variable
# Extract the table part from the markdown content
lines = content.split('\n')
# Find the header and separator lines
header_line = None
separator_line = None
for i, line in enumerate(lines):
    if '| Platform-show | Platform-name | rType | Domain |' in line:
        header_line = i
    elif '|---|---|---|---|---' in line:
        separator_line = i

if header_line is not None and separator_line is not None:
    table_lines = lines[header_line:separator_line] + lines[separator_line+1:]
    # Join the table lines and replace ' | ' with ',' for CSV parsing
    table_string = '\n'.join(table_lines).replace(' | ', ',').replace('|', '')
    # Read the string into a DataFrame
    mapping_df = pd.read_csv(StringIO(table_string))
    # Clean up column names by stripping whitespace
    mapping_df.columns = mapping_df.columns.str.strip()
    # Drop the 'หมายเหตุ' column if it's all NaN or empty after stripping
    if 'หมายเหตุ' in mapping_df.columns:
        mapping_df = mapping_df.drop(columns=['หมายเหตุ'])

    print("Mapping DataFrame created successfully:")
    display(mapping_df.head())
else:
    print("Could not find the table in the markdown content.")

Mapping DataFrame created successfully:


,Platform-show,Platform-name,rType,Domain
0,Cell Press eJournals,Cell Press,ARTICLE,www.cell.com
1,JCR,JCR,SEARCH,jcr.clarivate.com
2,JCR,JCR,ANALYSIS,jcr.clarivate.com
3,JSTOR eJournals,JSTOR,ARTICLE,www.jstor.org
4,JSTOR eJournals,JSTOR,BOOK_SECTION,www.jstor.org


In [8]:
print("Full DataFrame `df`:")
display(df)

Full DataFrame `df`:


,datetime,date,login,platform_name,publisher_name,rtype,mime,print_identifier,online_identifier,title_id,...,pii,sudoc-ppn,title,type,subject,geoip-country,geoip-latitude,geoip-longitude,host,url
0,2026-08-01T00:01:00+07:00,2026-08-01,Thaneeya.nan,Science Direct,NaN,ARTICLE,PDF,NaN,NaN,NaN,...,S0378517325001279,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.49.235.38,https://www.sciencedirect.com:443/science/arti...
1,2026-08-01T00:01:01+07:00,2026-08-01,Thaneeya.nan,Science Direct,NaN,ARTICLE,PDF,NaN,NaN,NaN,...,S0378517325001279,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.49.235.38,https://www.sciencedirect.com:443/science/arti...
2,2026-08-01T00:08:45+07:00,2026-08-01,u6506059,Nature Publishing Group,NaN,ARTICLE,PDF,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,223.207.219.216,https://www.nature.com:443/articles/s41419-024...
3,2026-08-01T00:08:46+07:00,2026-08-01,detchara.sum,Scopus,NaN,SEARCH,HTML,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.145.158.87,https://www.scopus.com:443/pages/search/author...
4,2026-08-01T00:08:47+07:00,2026-08-01,detchara.sum,Scopus,NaN,SEARCH,HTML,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.145.158.87,https://www.scopus.com:443/gateway/search-mana...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4698,2026-08-03T23:58:38+07:00,2026-08-03,sattawut.won,Science Direct,NaN,ARTICLE,HTML,NaN,NaN,NaN,...,S1052305712003357,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101.51.71.194,https://www.sciencedirect.com:443/science/arti...
4699,2026-08-03T23:58:39+07:00,2026-08-03,sattawut.won,Science Direct,NaN,ARTICLE,HTML,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101.51.71.194,https://www.sciencedirect.com:443/article/rela...
4700,2026-08-03T23:58:39+07:00,2026-08-03,sattawut.won,Science Direct,NaN,ARTICLE,HTML,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,159.192.143.54,https://www.sciencedirect.com:443/sdfe/arp/iss...
4701,2026-08-03T23:58:41+07:00,2026-08-03,sattawut.won,Science Direct,NaN,ARTICLE,PDF,NaN,NaN,NaN,...,S1052305712003357,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101.51.71.194,https://www.sciencedirect.com:443/science/arti...


In [9]:
print("Full DataFrame `mapping_df`:")
display(mapping_df)

Full DataFrame `mapping_df`:


,Platform-show,Platform-name,rType,Domain
0,Cell Press eJournals,Cell Press,ARTICLE,www.cell.com
1,JCR,JCR,SEARCH,jcr.clarivate.com
2,JCR,JCR,ANALYSIS,jcr.clarivate.com
3,JSTOR eJournals,JSTOR,ARTICLE,www.jstor.org
4,JSTOR eJournals,JSTOR,BOOK_SECTION,www.jstor.org
5,JSTOR eJournals,JSTOR,BOOK,www.jstor.org
6,Nature eJournals,Nature Publishing Group,ARTICLE,www.nature.com
7,SAGE eBooks,Sage Knowledge,BOOK_SECTION,sk.sagepub.com
8,SAGE eJournals,Sage Journals,ARTICLE,journals.sagepub.com
9,ScienceDirect eBooks,Science Direct,BOOK_SECTION,www.sciencedirect.com


In [10]:
import numpy as np

# 1. Strip whitespace from relevant columns in df to ensure accurate comparisons
df['domain'] = df['domain'].astype(str).str.strip()
df['rtype'] = df['rtype'].astype(str).str.strip()

# Standardize 'scifinder-n.cas.org' to 'scifinder.cas.org' in df['domain']
df['domain'] = df['domain'].replace('scifinder-n.cas.org', 'scifinder.cas.org')

# 2. Strip whitespace from mapping_df keys and value columns
mapping_df['Domain'] = mapping_df['Domain'].astype(str).str.strip()
mapping_df['rType'] = mapping_df['rType'].astype(str).str.strip()
mapping_df['Platform-show'] = mapping_df['Platform-show'].astype(str).str.strip()

# 3. Handle duplicates in mapping_df for (Domain, rType) combinations
#    If there are conflicting 'Platform-show' values for the same (Domain, rType), warn the user
#    and keep the first one.
duplicates = mapping_df[mapping_df.duplicated(subset=['Domain', 'rType'], keep=False)]

if not duplicates.empty:
    print("Warning: The following (Domain, rType) combinations are not unique in mapping_df:")
    display(duplicates.sort_values(by=['Domain', 'rType']))

    # Check if there are conflicting Platform-show for these duplicates
    conflicting_platforms = duplicates.groupby(['Domain', 'rType'])['Platform-show'].nunique() > 1
    conflicting_combinations = conflicting_platforms[conflicting_platforms].index.tolist()

    if conflicting_combinations:
        print("\nFurther Warning: The following (Domain, rType) combinations have conflicting 'Platform-show' values:")
        for domain, rtype in conflicting_combinations:
            print(f"- Domain: '{domain}', rType: '{rtype}'")
            display(duplicates[(duplicates['Domain'] == domain) & (duplicates['rType'] == rtype)])
        print("To resolve this, I will use the first 'Platform-show' listed for each conflicting combination.")
    else:
        print("\nAll duplicated (Domain, rType) combinations have consistent 'Platform-show' values.")

    mapping_df_cleaned = mapping_df.drop_duplicates(subset=['Domain', 'rType'], keep='first').copy()
    print("\nProceeding with a cleaned mapping_df where (Domain, rType) combinations are unique.")
else:
    print("(Domain, rType) is unique in mapping_df. Proceeding with mapping.")
    mapping_df_cleaned = mapping_df.copy()

# 4. Create the mapping dictionary: (Domain, rType) -> Platform-show
domain_rtype_to_platform_show_map = mapping_df_cleaned.set_index(['Domain', 'rType'])['Platform-show'].to_dict()

# 5. Create a combined key in df for mapping lookup
df['domain_rtype_key'] = list(zip(df['domain'], df['rtype']))

# 6. Apply the mapping to create the new 'Platform-show' column in df
df['Platform-show'] = df['domain_rtype_key'].map(domain_rtype_to_platform_show_map)

# 7. Filter df to keep only rows where 'Platform-show' was successfully mapped
initial_rows = df.shape[0]
df = df.dropna(subset=['Platform-show']).copy()
filtered_rows = df.shape[0]

print(f"\nFiltered df: Removed {initial_rows - filtered_rows} rows where 'Platform-show' could not be mapped.")
print(f"New df shape: {df.shape}")

# Display results of the filtered DataFrame
print("\nFirst 5 rows of the filtered df with the new 'Platform-show' column:")
display(df[['domain', 'rtype', 'Platform-show']].head())

# 8. Drop the temporary key column
df.drop(columns=['domain_rtype_key'], inplace=True, errors='ignore')

(Domain, rType) is unique in mapping_df. Proceeding with mapping.

Filtered df: Removed 0 rows where 'Platform-show' could not be mapped.
New df shape: (4703, 37)

First 5 rows of the filtered df with the new 'Platform-show' column:


,domain,rtype,Platform-show
0,www.sciencedirect.com,ARTICLE,ScienceDirect eJournals
1,www.sciencedirect.com,ARTICLE,ScienceDirect eJournals
2,www.nature.com,ARTICLE,Nature eJournals
3,www.scopus.com,SEARCH,Scopus
4,www.scopus.com,SEARCH,Scopus


In [ ]:
unique_platform_shows = df['Platform-show'].nunique()
print(f"Number of unique 'Platform-show' values: {unique_platform_shows}")

print("\nUnique 'Platform-show' values:")
display(df['Platform-show'].unique())

Number of unique 'Platform-show' values: 9

Unique 'Platform-show' values:


array(['ScienceDirect eJournals', 'Nature eJournals', 'Scopus',
       'SpringerLink eJournals', 'JCR', 'SpringerLink eBooks',
       "O'Reilly Learning", 'Cambridge eJournals', 'ScienceDirect eBooks'],
      dtype=object)

In [ ]:
unique_platform_shows_in_df = set(df['Platform-show'].unique())
unique_platform_shows_in_mapping = set(mapping_df['Platform-show'].unique())

missing_platform_shows = unique_platform_shows_in_mapping - unique_platform_shows_in_df

if missing_platform_shows:
    print("\n'Platform-show' values present in mapping_df but not in the current df:")
    for show in sorted(list(missing_platform_shows)):
        print(f"- {show}")
else:
    print("All 'Platform-show' values from mapping_df are present in the current df.")


'Platform-show' values present in mapping_df but not in the current df:
- Cambridge eBooks
- Cell Press eJournals
- JSTOR eJournals
- SAGE eBooks
- SAGE eJournals
- SciFinder
- Taylor & Francis eJournals
- Wiley eBooks
- Wiley eJournals


In [ ]:
if 'missing_platform_shows' not in locals():
    unique_platform_shows_in_df = set(df['Platform-show'].unique())
    unique_platform_shows_in_mapping = set(mapping_df['Platform-show'].unique())
    missing_platform_shows = unique_platform_shows_in_mapping - unique_platform_shows_in_df

print("Let's investigate why the following 'Platform-show' values are present in 'mapping_df' but not in your main DataFrame ('df'):")
for show in sorted(list(missing_platform_shows)):
    print(f"- {show}")

print("\n--- Detailed Reasons ---")

# Reason for 'Wiley eJournals'
if 'Wiley eJournals' in missing_platform_shows:
    print("\n**1. Wiley eJournals:**")
    print("The `mapping_df` entry for 'Wiley eJournals' shows an empty 'Domain' value:")
    display(mapping_df_cleaned[mapping_df_cleaned['Platform-show'] == 'Wiley eJournals'][['Domain', 'rType']])
    print("Since the `domain` column in your `df` contains actual domain names (not empty strings), no rows in `df` could match this specific mapping rule. Therefore, 'Wiley eJournals' was never assigned to any rows, and those potential rows were filtered out.")

# Reason for 'SAGE eBooks' and 'Wiley eBooks'
other_missing_platforms = [p for p in missing_platform_shows if p not in ['Wiley eJournals']]
if other_missing_platforms:
    print(f"\n**2. {', '.join(other_missing_platforms)}:**")
    print("For these 'Platform-show' values, the `mapping_df` entries have defined 'Domain' and 'rType' values. Their absence in your filtered `df` indicates that there were no rows in the *original* `df` that contained the specific `(domain, rtype)` combinations required to map to these platforms.")
    print("In other words, your raw usage data in `df` (before mapping and filtering) did not contain records corresponding to these specific platforms as defined by their 'Domain' and 'rType' in `mapping_df`.")
    print("\nRelevant mapping entries:")
    for platform_show in other_missing_platforms:
        print(f"  - For '{platform_show}':")
        display(mapping_df_cleaned[mapping_df_cleaned['Platform-show'] == platform_show][['Domain', 'rType']])

Let's investigate why the following 'Platform-show' values are present in 'mapping_df' but not in your main DataFrame ('df'):
- Cambridge eBooks
- Cell Press eJournals
- JSTOR eJournals
- SAGE eBooks
- SAGE eJournals
- SciFinder
- Taylor & Francis eJournals
- Wiley eBooks
- Wiley eJournals

--- Detailed Reasons ---

**1. Wiley eJournals:**
The `mapping_df` entry for 'Wiley eJournals' shows an empty 'Domain' value:


,Domain,rType
19,onlinelibrary.wiley.com,ARTICLE
20,acsess.onlinelibrary.wiley.com,ARTICLE


Since the `domain` column in your `df` contains actual domain names (not empty strings), no rows in `df` could match this specific mapping rule. Therefore, 'Wiley eJournals' was never assigned to any rows, and those potential rows were filtered out.

**2. SAGE eBooks, Wiley eBooks, Cell Press eJournals, Cambridge eBooks, SciFinder, JSTOR eJournals, Taylor & Francis eJournals, SAGE eJournals:**
For these 'Platform-show' values, the `mapping_df` entries have defined 'Domain' and 'rType' values. Their absence in your filtered `df` indicates that there were no rows in the *original* `df` that contained the specific `(domain, rtype)` combinations required to map to these platforms.
In other words, your raw usage data in `df` (before mapping and filtering) did not contain records corresponding to these specific platforms as defined by their 'Domain' and 'rType' in `mapping_df`.

Relevant mapping entries:
  - For 'SAGE eBooks':


,Domain,rType
7,sk.sagepub.com,BOOK_SECTION


  - For 'Wiley eBooks':


...

,Domain,rType
3,www.jstor.org,ARTICLE
4,www.jstor.org,BOOK_SECTION
5,www.jstor.org,BOOK


  - For 'Taylor & Francis eJournals':


,Domain,rType
16,www.tandfonline.com,ARTICLE


  - For 'SAGE eJournals':


,Domain,rType
8,journals.sagepub.com,ARTICLE


In [ ]:
print(f"Original df shape: {df.shape}")

# Separate DataFrames for ScienceDirect eJournals and Cell Press eJournals
sd_ejournal_df = df[df['Platform-show'] == 'ScienceDirect eJournals'].copy()
cell_press_ejournal_df = df[df['Platform-show'] == 'Cell Press eJournals'].copy()

print(f"\nShape of ScienceDirect eJournals: {sd_ejournal_df.shape}")
print(f"Shape of Cell Press eJournals: {cell_press_ejournal_df.shape}")

# Get unique titles from both DataFrames, handling potential NaN values
sd_titles = sd_ejournal_df['title'].dropna().unique()
cell_press_titles = cell_press_ejournal_df['title'].dropna().unique()

# Find titles that are common to both
common_titles = pd.Series(sd_titles)[pd.Series(sd_titles).isin(cell_press_titles)].tolist()

print(f"\nNumber of common titles found: {len(common_titles)}")
if len(common_titles) > 0:
    print("Sample of common titles:")
    display(pd.Series(common_titles).head())
else:
    print("No common titles found between ScienceDirect eJournals and Cell Press eJournals.")

# Filter out records from sd_ejournal_df that have titles present in common_titles
sd_ejournal_df_filtered = sd_ejournal_df[~sd_ejournal_df['title'].isin(common_titles)]

print(f"\nNumber of ScienceDirect eJournals records removed: {sd_ejournal_df.shape[0] - sd_ejournal_df_filtered.shape[0]}")

# Combine the filtered ScienceDirect eJournals with Cell Press eJournals and other platforms
other_platforms_df = df[~df['Platform-show'].isin(['ScienceDirect eJournals', 'Cell Press eJournals'])]

# Reconstruct the main DataFrame
df = pd.concat([sd_ejournal_df_filtered, cell_press_ejournal_df, other_platforms_df])

print(f"\nNew df shape after filtering: {df.shape}")
print("First 5 rows of the updated df:")
display(df.head())

Original df shape: (4703, 36)

Shape of ScienceDirect eJournals: (2198, 36)
Shape of Cell Press eJournals: (0, 36)

Number of common titles found: 0
No common titles found between ScienceDirect eJournals and Cell Press eJournals.

Number of ScienceDirect eJournals records removed: 0

New df shape after filtering: (4703, 36)
First 5 rows of the updated df:


,datetime,date,login,platform_name,publisher_name,rtype,mime,print_identifier,online_identifier,title_id,...,sudoc-ppn,title,type,subject,geoip-country,geoip-latitude,geoip-longitude,host,url,Platform-show
0,2026-08-01T00:01:00+07:00,2026-08-01,Thaneeya.nan,Science Direct,NaN,ARTICLE,PDF,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.49.235.38,https://www.sciencedirect.com:443/science/arti...,ScienceDirect eJournals
1,2026-08-01T00:01:01+07:00,2026-08-01,Thaneeya.nan,Science Direct,NaN,ARTICLE,PDF,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.49.235.38,https://www.sciencedirect.com:443/science/arti...,ScienceDirect eJournals
7,2026-08-01T00:29:36+07:00,2026-08-01,g6737504,Science Direct,NaN,ARTICLE,HTML,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,110.168.219.147,https://www.sciencedirect.com:443/science/arti...,ScienceDirect eJournals
8,2026-08-01T00:29:37+07:00,2026-08-01,g6737504,Science Direct,NaN,ARTICLE,HTML,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,110.168.219.147,https://www.sciencedirect.com:443/article/rela...,ScienceDirect eJournals
9,2026-08-01T00:29:37+07:00,2026-08-01,g6737504,Science Direct,NaN,ARTICLE,HTML,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,110.168.219.147,https://www.sciencedirect.com:443/sdfe/arp/iss...,ScienceDirect eJournals


In [ ]:
if 'platform_name' in df.columns:
    df = df.drop(columns=['platform_name'])
    print("'platform_name' column removed.")
else:
    print("'platform_name' column not found in DataFrame.")

print(f"Current df shape: {df.shape}")
display(df.head())

'platform_name' column removed.
Current df shape: (4703, 35)


,datetime,date,login,publisher_name,rtype,mime,print_identifier,online_identifier,title_id,doi,...,sudoc-ppn,title,type,subject,geoip-country,geoip-latitude,geoip-longitude,host,url,Platform-show
0,2026-08-01T00:01:00+07:00,2026-08-01,Thaneeya.nan,NaN,ARTICLE,PDF,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.49.235.38,https://www.sciencedirect.com:443/science/arti...,ScienceDirect eJournals
1,2026-08-01T00:01:01+07:00,2026-08-01,Thaneeya.nan,NaN,ARTICLE,PDF,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.49.235.38,https://www.sciencedirect.com:443/science/arti...,ScienceDirect eJournals
7,2026-08-01T00:29:36+07:00,2026-08-01,g6737504,NaN,ARTICLE,HTML,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,110.168.219.147,https://www.sciencedirect.com:443/science/arti...,ScienceDirect eJournals
8,2026-08-01T00:29:37+07:00,2026-08-01,g6737504,NaN,ARTICLE,HTML,NaN,NaN,NaN,10.1067/j.cpradiol.2017.12.001/2019-04-12T14:1...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,110.168.219.147,https://www.sciencedirect.com:443/article/rela...,ScienceDirect eJournals
9,2026-08-01T00:29:37+07:00,2026-08-01,g6737504,NaN,ARTICLE,HTML,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,110.168.219.147,https://www.sciencedirect.com:443/sdfe/arp/iss...,ScienceDirect eJournals


In [ ]:
output_csv_file_path = '/content/08-2026_clean.csv'
df.to_csv(output_csv_file_path, index=False)
print(f"Cleaned data saved to: {output_csv_file_path}")

Cleaned data saved to: /content/08-2026_clean.csv


In [13]:
ezpa_csv_file_path = '/content/drive/MyDrive/EZproxyData/ezpa_aug01_03.csv'
ezpa_df = pd.read_csv(ezpa_csv_file_path, sep=';', on_bad_lines='warn', engine='python')
display(ezpa_df.head())

,datetime,date,login,platform,platform_name,publisher_name,rtype,mime,print_identifier,online_identifier,...,type,subject,geoip-country,geoip-latitude,geoip-longitude,host,identd,url,status,size
0,2026-07-31T17:00:56+00:00,2026-07-31,585cbd88f57a8949745aa1dc59243b586e10b5fd,sd,Science Direct,Elsevier,SEARCH,HTML,NaN,NaN,...,NaN,NaN,TH,13.7512,100.5172,7edfd483b00ddcff05aeae5c0e8a00cf4a63dafd,NaN,https://www.sciencedirect.com:443/search?qs=Ru...,200,97161
1,2026-07-31T17:01:01+00:00,2026-07-31,585cbd88f57a8949745aa1dc59243b586e10b5fd,sd,Science Direct,Elsevier,ARTICLE,PDF,0378-5173,0378-5173,...,journal-article,NaN,TH,13.7512,100.5172,7edfd483b00ddcff05aeae5c0e8a00cf4a63dafd,NaN,https://www.sciencedirect.com:443/science/arti...,200,7544
2,2026-07-31T17:01:48+00:00,2026-07-31,eca915a817080f18a9b15c50fa5b75acb40b6e4a,scifinder-n,Scifinder-n,NaN,SEARCH,HTML,NaN,NaN,...,NaN,NaN,IN,17.3724,78.4378,ad4acf84adcce5461ac2d1aa283e82c6037ac534,NaN,https://scifinder-n.cas.org:443/internal/api/s...,200,540
3,2026-07-31T17:01:48+00:00,2026-07-31,eca915a817080f18a9b15c50fa5b75acb40b6e4a,scifinder-n,Scifinder-n,NaN,SEARCH,HTML,NaN,NaN,...,NaN,NaN,IN,17.3724,78.4378,ad4acf84adcce5461ac2d1aa283e82c6037ac534,NaN,https://scifinder-n.cas.org:443/internal/api/s...,200,391052
4,2026-07-31T17:01:53+00:00,2026-07-31,eca915a817080f18a9b15c50fa5b75acb40b6e4a,scifinder-n,Scifinder-n,NaN,SEARCH,HTML,NaN,NaN,...,NaN,NaN,IN,17.3724,78.4378,ad4acf84adcce5461ac2d1aa283e82c6037ac534,NaN,https://scifinder-n.cas.org:443/internal/api/s...,200,3764020


In [14]:
ezpa_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 36 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   datetime             8068 non-null   object 
 1   date                 8068 non-null   object 
 2   login                8068 non-null   object 
 3   platform             8068 non-null   object 
 4   platform_name        8068 non-null   object 
 5   publisher_name       4422 non-null   object 
 6   rtype                8065 non-null   object 
 7   mime                 8060 non-null   object 
 8   print_identifier     3384 non-null   object 
 9   online_identifier    3163 non-null   object 
 10  title_id             3632 non-null   object 
 11  doi                  3031 non-null   object 
 12  publication_title    3404 non-null   object 
 13  publication_date     3323 non-null   float64
 14  unitid               5747 non-null   object 
 15  domain               8068 non-null   o